In [0]:
events = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv", header = True, inferSchema = True)
events.describe(["price"]).display()

summary,price
count,42448764
mean,290.3236606849198
stddev,358.2691553394021
min,0.0
max,2574.07


In [0]:
import pyspark.sql.functions as F

weekday = events.withColumn("is_weekend",
    F.dayofweek("event_time").isin([1,7]))
weekday.groupBy("is_weekend", "event_type").count().display()

is_weekend,event_type,count
false,purchase,546439
false,view,29775216
false,cart,664318
true,view,11004183
true,cart,262198
true,purchase,196410


In [0]:
events.stat.corr("price", "user_id")

0.0033993499465154067

In [0]:
from pyspark.sql.window import Window

features = events.withColumn("hour", F.hour("event_time")) \
    .withColumn("day_of_week", F.dayofweek("event_time")) \
    .withColumn("price_log", F.log(F.col("price")+1)) \
    .withColumn("time_since_first_view",
        F.unix_timestamp("event_time") -
        F.unix_timestamp(F.first("event_time").over(Window.partitionBy("user_id").orderBy("event_time"))))
    
display(features)    

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,hour,day_of_week,price_log,time_since_first_view
2019-10-07T06:23:01.000Z,view,16200119,2053013556344914381,kids.fmcg.diapers,moony,18.47,222907508,cb653adc-46a2-4d90-9e34-5bdfb2be30ce,6,2,2.968874819384108,0
2019-10-07T06:26:23.000Z,view,16200162,2053013556344914381,kids.fmcg.diapers,moony,18.47,222907508,cb653adc-46a2-4d90-9e34-5bdfb2be30ce,6,2,2.968874819384108,202
2019-10-08T14:29:09.000Z,view,6200883,2053013552293216471,appliances.environment.air_heater,elenberg,46.31,244673419,e2f0524c-bfc4-4c69-b93a-56f983027af3,14,3,3.8567216896430567,0
2019-10-12T10:15:48.000Z,view,17300355,2053013553853497655,null,creed,240.16,257849716,71e76013-465a-4644-b82f-ab7fc64c9e95,10,7,5.485460613621205,0
2019-10-22T22:05:40.000Z,view,3900896,2053013552326770905,appliances.environment.water_heater,klima,77.2,266203246,c83d6f3d-2973-411f-8180-e476e65bc54c,22,3,4.359269647551265,0
2019-10-24T01:14:36.000Z,view,3900896,2053013552326770905,appliances.environment.water_heater,klima,77.2,266203246,56944410-059f-4f08-939b-867b3e060741,1,5,4.359269647551265,97736
2019-10-06T11:29:22.000Z,view,22700574,2053013556168753601,null,null,88.81,278272605,e4cd7037-61d8-461c-ba6f-4314f0fb9a6f,11,1,4.497696327682858,0
2019-10-06T11:30:58.000Z,view,22700129,2053013556168753601,null,stels,66.93,278272605,0a20874c-c88c-4628-a6eb-886784b61d19,11,1,4.218477763203211,96
2019-10-06T11:31:53.000Z,view,22700078,2053013556168753601,null,stels,180.18,278272605,0a20874c-c88c-4628-a6eb-886784b61d19,11,1,5.1994910122411415,151
2019-10-06T11:34:45.000Z,view,22700078,2053013556168753601,null,stels,180.18,278272605,0a20874c-c88c-4628-a6eb-886784b61d19,11,1,5.1994910122411415,323
